# Post-Accident Guidance Assistant — LLM-as-Judge Evaluation

**Goal:** Measure how accurately and clearly our AI recommends claim vs. no-claim decisions.

**Method:**
1. Run 20 golden test cases (known correct answers) through the live API
2. Check decision accuracy (rule-verifiable)
3. Use Groq LLM as a judge to score reasoning quality (1–5)
4. Produce a final scorecard

**Categories tested:**
- `repair_below_deductible` → must always be `DO_NOT_CLAIM`
- `injuries_present` → must always be `CLAIM`
- `rideshare_vehicle` → must always be `CONSULT_AGENT`
- `prior_claims` → must always be `CONSULT_AGENT`
- `math_claim` → financially worth claiming
- `math_no_claim` → financially NOT worth claiming
- `no_fault_state` → different math applies

In [ ]:
!pip install requests groq pandas tabulate -q

In [ ]:
import os, json, time, requests
from groq import Groq
import pandas as pd

# ── Config ──────────────────────────────────────────────────────────────────
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")
API_URL      = "https://post-accident-guidance-assistant-production.up.railway.app/chat"
JUDGE_MODEL  = "llama-3.3-70b-versatile"

groq_client = Groq(api_key=GROQ_API_KEY)
print("Config loaded. API:", API_URL)

In [ ]:
# ── Golden Test Cases ────────────────────────────────────────────────────────
# Each case has a deterministic expected_decision based on our rule engine logic.
# Math verification shown in comments for transparency.

GOLDEN_CASES = [
    # ── Category: repair_below_deductible (always DO_NOT_CLAIM) ──
    {
        "id": "DNF-001",
        "category": "repair_below_deductible",
        "description": "Repair ($200) < deductible ($500)",
        "scenario": {"repair_cost": 200, "deductible": 500, "annual_premium": 1200,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "CA", "notes": "Small scratch on bumper"},
        "expected_decision": "DO_NOT_CLAIM"
    },
    {
        "id": "DNF-002",
        "category": "repair_below_deductible",
        "description": "Repair ($300) < deductible ($1000)",
        "scenario": {"repair_cost": 300, "deductible": 1000, "annual_premium": 1800,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "TX", "notes": "Parking lot dent"},
        "expected_decision": "DO_NOT_CLAIM"
    },
    {
        "id": "DNF-003",
        "category": "repair_below_deductible",
        "description": "Repair ($500) == deductible ($500)",
        "scenario": {"repair_cost": 500, "deductible": 500, "annual_premium": 1400,
                     "at_fault": False, "injuries": False, "prior_claims_2yr": 0,
                     "state": "OH", "notes": "Shopping cart damage"},
        "expected_decision": "DO_NOT_CLAIM"
    },

    # ── Category: injuries_present (always CLAIM) ──
    {
        "id": "CLM-001",
        "category": "injuries_present",
        "description": "Injuries override everything → CLAIM",
        "scenario": {"repair_cost": 1500, "deductible": 2000, "annual_premium": 1200,
                     "at_fault": True, "injuries": True, "prior_claims_2yr": 0,
                     "state": "CA", "notes": "Passenger complained of neck pain"},
        "expected_decision": "CLAIM"
    },
    {
        "id": "CLM-002",
        "category": "injuries_present",
        "description": "Injuries even when repair < deductible → CLAIM",
        "scenario": {"repair_cost": 300, "deductible": 500, "annual_premium": 1600,
                     "at_fault": False, "injuries": True, "prior_claims_2yr": 0,
                     "state": "NY", "notes": "Rear-ended, driver has back pain"},
        "expected_decision": "CLAIM"
    },
    {
        "id": "CLM-003",
        "category": "injuries_present",
        "description": "Serious injuries, major repair → CLAIM",
        "scenario": {"repair_cost": 12000, "deductible": 1000, "annual_premium": 2000,
                     "at_fault": True, "injuries": True, "prior_claims_2yr": 0,
                     "state": "TX", "notes": "Multi-car freeway accident, ambulance called"},
        "expected_decision": "CLAIM"
    },

    # ── Category: rideshare_vehicle (always CONSULT_AGENT) ──
    {
        "id": "CA-001",
        "category": "rideshare_vehicle",
        "description": "Uber mention in notes → CONSULT_AGENT",
        "scenario": {"repair_cost": 3000, "deductible": 500, "annual_premium": 1400,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "CA", "notes": "Was driving for Uber when accident happened"},
        "expected_decision": "CONSULT_AGENT"
    },
    {
        "id": "CA-002",
        "category": "rideshare_vehicle",
        "description": "Lyft mention → CONSULT_AGENT",
        "scenario": {"repair_cost": 5000, "deductible": 500, "annual_premium": 1200,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "FL", "notes": "Lyft driver, had passenger in car"},
        "expected_decision": "CONSULT_AGENT"
    },
    {
        "id": "CA-003",
        "category": "rideshare_vehicle",
        "description": "Classic car mention → CONSULT_AGENT",
        "scenario": {"repair_cost": 4000, "deductible": 500, "annual_premium": 1800,
                     "at_fault": False, "injuries": False, "prior_claims_2yr": 0,
                     "state": "TX", "notes": "1969 classic Mustang, restored"},
        "expected_decision": "CONSULT_AGENT"
    },

    # ── Category: prior_claims (always CONSULT_AGENT) ──
    {
        "id": "CA-004",
        "category": "prior_claims",
        "description": "1 prior claim → CONSULT_AGENT",
        "scenario": {"repair_cost": 5000, "deductible": 500, "annual_premium": 1600,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 1,
                     "state": "GA", "notes": "Second accident this year"},
        "expected_decision": "CONSULT_AGENT"
    },
    {
        "id": "CA-005",
        "category": "prior_claims",
        "description": "2 prior claims → CONSULT_AGENT",
        "scenario": {"repair_cost": 8000, "deductible": 1000, "annual_premium": 2200,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 2,
                     "state": "NY", "notes": "Third incident in two years"},
        "expected_decision": "CONSULT_AGENT"
    },

    # ── Category: math_claim (net > 3yr premium cost → CLAIM) ──
    # net=6500, increase=1200*0.40=480/yr → 1440 over 3yr → 6500 > 1440 → CLAIM
    {
        "id": "MC-001",
        "category": "math_claim",
        "description": "net=$6500 >> 3yr cost=$1440 → CLAIM",
        "scenario": {"repair_cost": 7000, "deductible": 500, "annual_premium": 1200,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "WA", "notes": "Hit guardrail on highway"},
        "expected_decision": "CLAIM"
    },
    # net=4000, increase=1800*0.15=270/yr → 810 over 3yr → 4000 > 810 → CLAIM
    {
        "id": "MC-002",
        "category": "math_claim",
        "description": "Not at fault: net=$4000 >> 3yr cost=$810 → CLAIM",
        "scenario": {"repair_cost": 5000, "deductible": 1000, "annual_premium": 1800,
                     "at_fault": False, "injuries": False, "prior_claims_2yr": 0,
                     "state": "AZ", "notes": "Other driver ran red light"},
        "expected_decision": "CLAIM"
    },
    # net=8500, increase=2000*0.40=800/yr → 2400 over 3yr → 8500 > 2400 → CLAIM
    {
        "id": "MC-003",
        "category": "math_claim",
        "description": "Major repair: net=$8500 >> 3yr cost=$2400 → CLAIM",
        "scenario": {"repair_cost": 9500, "deductible": 1000, "annual_premium": 2000,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "CO", "notes": "Deer collision, significant body damage"},
        "expected_decision": "CLAIM"
    },

    # ── Category: math_no_claim (net < 3yr premium cost → DO_NOT_CLAIM) ──
    # net=1500, increase=4000*0.40=1600/yr → 4800 over 3yr → 1500 < 4800 → DO_NOT_CLAIM
    {
        "id": "MNC-001",
        "category": "math_no_claim",
        "description": "High premium: net=$1500 << 3yr cost=$4800 → DO_NOT_CLAIM",
        "scenario": {"repair_cost": 2000, "deductible": 500, "annual_premium": 4000,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "CA", "notes": "Minor fender bender, already paying high premium"},
        "expected_decision": "DO_NOT_CLAIM"
    },
    # net=1000, increase=3000*0.40=1200/yr → 3600 over 3yr → 1000 < 3600 → DO_NOT_CLAIM
    {
        "id": "MNC-002",
        "category": "math_no_claim",
        "description": "High premium: net=$1000 << 3yr cost=$3600 → DO_NOT_CLAIM",
        "scenario": {"repair_cost": 1500, "deductible": 500, "annual_premium": 3000,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "IL", "notes": "Backed into a pole"},
        "expected_decision": "DO_NOT_CLAIM"
    },

    # ── Category: no_fault_state (10% premium increase instead of 40%) ──
    # Michigan: net=3500, increase=1800*0.10=180/yr → 540 over 3yr → 3500 > 540 → CLAIM
    {
        "id": "NF-001",
        "category": "no_fault_state",
        "description": "Michigan (no-fault): lower increase means CLAIM is worthwhile",
        "scenario": {"repair_cost": 4000, "deductible": 500, "annual_premium": 1800,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "MI", "notes": "Intersection collision in Detroit"},
        "expected_decision": "CLAIM"
    },
    # New Jersey: net=2500, increase=1400*0.10=140/yr → 420 over 3yr → 2500 > 420 → CLAIM
    {
        "id": "NF-002",
        "category": "no_fault_state",
        "description": "New Jersey (no-fault): PIP applies, CLAIM recommended",
        "scenario": {"repair_cost": 3000, "deductible": 500, "annual_premium": 1400,
                     "at_fault": True, "injuries": False, "prior_claims_2yr": 0,
                     "state": "NJ", "notes": "Rear-ended at traffic light"},
        "expected_decision": "CLAIM"
    },
]

print(f"Loaded {len(GOLDEN_CASES)} golden test cases across {len(set(c['category'] for c in GOLDEN_CASES))} categories")

In [ ]:
# ── API Runner ───────────────────────────────────────────────────────────────
def call_api(case: dict) -> dict:
    payload = {
        "user_id": f"eval-{case['id']}",
        "message": "Please analyze my accident scenario.",
        "scenario": case["scenario"]
    }
    start = time.time()
    resp = requests.post(API_URL, json=payload, timeout=30)
    latency = round((time.time() - start) * 1000)
    resp.raise_for_status()
    data = resp.json()
    return {
        "decision": data["recommendation"]["decision"],
        "confidence": data["recommendation"]["confidence"],
        "confidence_score": data["recommendation"]["confidence_score"],
        "summary": data["recommendation"]["summary"],
        "reasoning": data["recommendation"]["reasoning"],
        "financial_breakdown": data["recommendation"]["financial_breakdown"],
        "key_factors": data["recommendation"]["key_factors"],
        "next_steps": data["recommendation"]["next_steps"],
        "latency_ms": latency,
    }

print("API runner ready.")

In [ ]:
# ── Run All Evaluations ──────────────────────────────────────────────────────
results = []

for i, case in enumerate(GOLDEN_CASES):
    try:
        api_result = call_api(case)
        passed = api_result["decision"] == case["expected_decision"]
        results.append({
            "id": case["id"],
            "category": case["category"],
            "description": case["description"],
            "expected": case["expected_decision"],
            "actual": api_result["decision"],
            "passed": passed,
            "confidence": api_result["confidence"],
            "confidence_score": api_result["confidence_score"],
            "latency_ms": api_result["latency_ms"],
            "reasoning": api_result["reasoning"],
            "financial_breakdown": api_result["financial_breakdown"],
            "key_factors": api_result["key_factors"],
            "next_steps": api_result["next_steps"],
            "scenario": case["scenario"],
        })
        status = "PASS" if passed else "FAIL"
        print(f"[{i+1:02d}] {case['id']:<8} {status}  expected={case['expected_decision']:<15} actual={api_result['decision']:<15} ({api_result['latency_ms']}ms)")
    except Exception as e:
        print(f"[{i+1:02d}] {case['id']:<8} ERROR: {e}")
        results.append({
            "id": case["id"], "category": case["category"],
            "description": case["description"], "expected": case["expected_decision"],
            "actual": "ERROR", "passed": False, "confidence": None,
            "confidence_score": None, "latency_ms": None,
            "reasoning": "", "financial_breakdown": "", "key_factors": [], "next_steps": [],
            "scenario": case["scenario"],
        })
    time.sleep(0.8)

total = len(results)
passed_count = sum(1 for r in results if r["passed"])
print(f"\n{'='*60}")
print(f"Decision Accuracy: {passed_count}/{total} = {passed_count/total*100:.1f}%")

In [ ]:
# ── Decision Accuracy by Category ────────────────────────────────────────────
df = pd.DataFrame(results)

accuracy_by_category = (
    df.groupby("category")["passed"]
    .agg(correct="sum", total="count")
    .assign(accuracy=lambda x: (x["correct"] / x["total"] * 100).round(1))
    .sort_values("accuracy", ascending=False)
)

print("\nAccuracy by Category:")
print(accuracy_by_category.to_string())

print("\nFailed cases:")
failed = df[~df["passed"]][["id", "category", "expected", "actual", "description"]]
if failed.empty:
    print("None! Perfect decision accuracy.")
else:
    print(failed.to_string(index=False))

In [ ]:
# ── LLM-as-Judge: Reasoning Quality Scorer ───────────────────────────────────
JUDGE_SYSTEM = """You are an expert evaluator assessing the quality of AI-generated auto insurance guidance.

You will receive:
- The accident scenario
- The AI's decision and reasoning
- The correct expected decision

Score the AI response on three dimensions (1-5 each):

1. ACCURACY (1-5): Does the financial breakdown correctly explain the math? Does the reasoning match the decision?
   1=completely wrong, 3=partially correct, 5=fully accurate

2. CLARITY (1-5): Is the explanation easy to understand for a non-expert driver?
   1=confusing jargon, 3=somewhat clear, 5=crystal clear and actionable

3. COMPLETENESS (1-5): Does it address the key factors (state law, financial math, special circumstances)?
   1=misses most factors, 3=covers main points, 5=thorough and covers all relevant factors

Return ONLY valid JSON:
{
  "accuracy_score": <1-5>,
  "clarity_score": <1-5>,
  "completeness_score": <1-5>,
  "overall_score": <average of three, 1 decimal>,
  "strengths": "<one sentence on what the response did well>",
  "weaknesses": "<one sentence on what could be improved, or 'None' if perfect>"
}"""


def judge_response(result: dict) -> dict:
    user_content = (
        f"SCENARIO: {json.dumps(result['scenario'], indent=2)}\n\n"
        f"EXPECTED DECISION: {result['expected']}\n"
        f"ACTUAL DECISION: {result['actual']}\n"
        f"DECISION CORRECT: {result['passed']}\n\n"
        f"FINANCIAL BREAKDOWN:\n{result['financial_breakdown']}\n\n"
        f"REASONING:\n{result['reasoning']}\n\n"
        f"KEY FACTORS: {result['key_factors']}\n"
        f"NEXT STEPS: {result['next_steps']}"
    )
    response = groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
        max_tokens=256,
    )
    return json.loads(response.choices[0].message.content)


print("Judge ready. Running quality scores...")

In [ ]:
# ── Run Judge on All Results ─────────────────────────────────────────────────
for i, result in enumerate(results):
    if result["actual"] == "ERROR":
        result["accuracy_score"] = result["clarity_score"] = result["completeness_score"] = result["overall_score"] = 0
        result["strengths"] = result["weaknesses"] = "API call failed"
        continue
    try:
        scores = judge_response(result)
        result.update(scores)
        print(f"[{i+1:02d}] {result['id']:<8} accuracy={scores['accuracy_score']} clarity={scores['clarity_score']} completeness={scores['completeness_score']} → overall={scores['overall_score']}")
    except Exception as e:
        print(f"[{i+1:02d}] {result['id']:<8} JUDGE ERROR: {e}")
        result["accuracy_score"] = result["clarity_score"] = result["completeness_score"] = result["overall_score"] = 0
        result["strengths"] = result["weaknesses"] = f"Judge error: {e}"
    time.sleep(0.5)

print("\nJudging complete.")

In [ ]:
# ── Final Scorecard ───────────────────────────────────────────────────────────
df = pd.DataFrame(results)

total          = len(df)
passed_count   = df["passed"].sum()
accuracy_pct   = passed_count / total * 100
avg_overall    = df["overall_score"].mean()
avg_accuracy   = df["accuracy_score"].mean()
avg_clarity    = df["clarity_score"].mean()
avg_complete   = df["completeness_score"].mean()
avg_latency    = df["latency_ms"].dropna().mean()

print("=" * 55)
print("  POST-ACCIDENT GUIDANCE ASSISTANT — EVAL SCORECARD")
print("=" * 55)
print(f"  Test Cases          : {total}")
print(f"  Decision Accuracy   : {passed_count}/{total} ({accuracy_pct:.1f}%)")
print()
print(f"  Reasoning Quality (LLM-as-Judge, out of 5):")
print(f"    Accuracy          : {avg_accuracy:.2f}")
print(f"    Clarity           : {avg_clarity:.2f}")
print(f"    Completeness      : {avg_complete:.2f}")
print(f"    Overall           : {avg_overall:.2f}")
print()
print(f"  Avg Latency         : {avg_latency:.0f}ms")
print("=" * 55)

print("\nAccuracy by Category:")
cat_summary = (
    df.groupby("category").agg(
        correct=("passed", "sum"),
        total=("passed", "count"),
        avg_quality=("overall_score", "mean"),
    )
    .assign(
        accuracy=lambda x: (x["correct"] / x["total"] * 100).map("{:.0f}%".format),
        avg_quality=lambda x: x["avg_quality"].map("{:.2f}".format),
    )
)
print(cat_summary.to_string())

In [ ]:
# ── Sample Analysis: Best and Worst Responses ─────────────────────────────────
df_scored = df[df["overall_score"] > 0].copy()

best  = df_scored.loc[df_scored["overall_score"].idxmax()]
worst = df_scored.loc[df_scored["overall_score"].idxmin()]

print("BEST RESPONSE")
print(f"  Case: {best['id']} ({best['category']})")
print(f"  Decision: {best['actual']} (expected {best['expected']}) — {'PASS' if best['passed'] else 'FAIL'}")
print(f"  Quality: {best['overall_score']} (accuracy={best['accuracy_score']}, clarity={best['clarity_score']}, completeness={best['completeness_score']})")
print(f"  Strengths: {best['strengths']}")
print(f"  Reasoning snippet: {best['reasoning'][:300]}...\n")

print("WORST RESPONSE")
print(f"  Case: {worst['id']} ({worst['category']})")
print(f"  Decision: {worst['actual']} (expected {worst['expected']}) — {'PASS' if worst['passed'] else 'FAIL'}")
print(f"  Quality: {worst['overall_score']} (accuracy={worst['accuracy_score']}, clarity={worst['clarity_score']}, completeness={worst['completeness_score']})")
print(f"  Weaknesses: {worst['weaknesses']}")
print(f"  Reasoning snippet: {worst['reasoning'][:300]}...")

In [ ]:
# ── Export Results ────────────────────────────────────────────────────────────
export_cols = ["id", "category", "description", "expected", "actual", "passed",
               "confidence", "confidence_score", "latency_ms",
               "accuracy_score", "clarity_score", "completeness_score", "overall_score",
               "strengths", "weaknesses"]

df[export_cols].to_csv("eval_results.csv", index=False)
print("Results saved to eval_results.csv")